# 03 - Bronze -> Silver

Aplica **renome de colunas** e **Data Quality**, e grava tabelas Delta MANAGED no schema `silver`.

| Etapa | O que faz |
|---|---|
| Nomes | colunas em MAIÚSCULAS, snake_case; prefixo regional `NA_`, `EU_`, `JP_`, `OTHER_` vira nome completo |
| Texto | `trim`; vazio, `tbd`, `N/A`, `null`... viram `NULL`; listas em texto (`['A', 'B']`) viram `A, B` |
| Tipos | colunas string 100% numéricas / datas são convertidas (`try_cast`, sem erro com ANSI ligado) |
| Faixas | `METASCORE` fora de 0-100 e `USER_SCORE` fora de 0-10 viram `NULL` |
| Linhas | descarta linhas sem chave (`ID`, `NAME`) e duplicadas |
| Auditoria | `FILENAME_BRONZE` (origem) e `FILEDATE_SILVER` (data/hora do processamento) |

In [ ]:
import re

from pyspark.sql import DataFrame, functions as F

CATALOG       = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Valores que significam "sem informação"
NULL_TOKENS = ["", "tbd", "n/a", "null", "none", "nan"]

# Expressões usadas para descobrir o tipo real de colunas que chegaram como string
INT_RE    = r"^[+-]?\d+$"
DOUBLE_RE = r"^[+-]?\d+(\.\d+)?$"
DATE_RE   = r"^\d{4}-\d{2}-\d{2}$"

NUMERIC_TYPES = ("int", "bigint", "smallint", "tinyint", "double", "float") 

# ---------- Regras de nomes de colunas ----------
REGION_PREFIXES = {
    "NA_":    "NORTH_AMERICA_",
    "EU_":    "EUROPE_",
    "JP_":    "JAPAN_",
    "OTHER_": "OTHER_REGIONAL_",
}

snake_upper_case = lambda x: re.sub(r"(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])", "_", x).upper()


def apply_name_rules(colname: str) -> str:
    """MAIÚSCULAS + snake_case; troca o PREFIXO regional (NA_, EU_, JP_, OTHER_) pelo nome completo."""
    n = snake_upper_case(colname.strip())
    n = re.sub(r"[^0-9A-Z]+", "_", n).strip("_")   # espaços/símbolos viram "_"
    for old, new in REGION_PREFIXES.items():
        if n.startswith(old):          # só no início: evita trocar "NA_" no meio de outra palavra
            return new + n[len(old):]
    return n


def _safe_drop(df: DataFrame, cols) -> DataFrame:
    """Dropa colunas somente se existirem."""
    existing = set(df.columns)
    to_drop = [c for c in cols if c in existing]
    return df.drop(*to_drop) if to_drop else df


# ---------- Regras de Data Quality ----------
def clean_list_like(df: DataFrame, cols) -> DataFrame:
    """Transforma "['A', 'B']" (lista escrita como texto) em "A, B"."""
    for c in cols:
        if c in df.columns:
            no_brackets = F.regexp_replace(F.col(c), r"^\[['\"]?|['\"]?\]$", "")
            df = df.withColumn(c, F.regexp_replace(no_brackets, r"['\"],\s*['\"]", ", "))
    return df


def clean_strings(df: DataFrame) -> DataFrame:
    """trim em todas as colunas texto e vazio/'tbd'/'N/A'... vira NULL."""
    for name, dtype in df.dtypes:
        if dtype == "string":
            trimmed = F.trim(F.col(name))
            df = df.withColumn(
                name,
                F.when(F.lower(trimmed).isin(*NULL_TOKENS), F.lit(None).cast("string")).otherwise(trimmed),
            )
    return df


def auto_cast_types(df: DataFrame) -> DataFrame:
    """
    Colunas string em que TODOS os valores não nulos são inteiros / decimais / datas (yyyy-MM-dd)
    são convertidas para o tipo correto. Usa try_cast: valor inválido vira NULL e não derruba o job
    (o serverless roda com ANSI ligado, onde um CAST inválido gera erro).
    """
    str_cols = [c for c, t in df.dtypes if t == "string"]
    if not str_cols:
        return df

    aggs = []
    for i, c in enumerate(str_cols):
        col = F.col(c)
        aggs += [
            F.count(col).alias(f"n_{i}"),
            F.count(F.when(col.rlike(INT_RE), 1)).alias(f"int_{i}"),
            F.count(F.when(col.rlike(DOUBLE_RE), 1)).alias(f"dbl_{i}"),
            F.count(F.when(col.rlike(DATE_RE), 1)).alias(f"date_{i}"),
            F.max(F.length(col)).alias(f"len_{i}"),
        ]
    stats = df.agg(*aggs).first().asDict()

    for i, c in enumerate(str_cols):
        n = stats[f"n_{i}"]
        if n == 0:
            continue
        if stats[f"int_{i}"] == n:
            target = "int" if stats[f"len_{i}"] <= 9 else "bigint"
        elif stats[f"dbl_{i}"] == n:
            target = "double"
        elif stats[f"date_{i}"] == n:
            target = "date"
        else:
            continue
        df = df.withColumn(c, F.expr(f"try_cast(`{c}` AS {target})"))
    return df


def apply_ranges(df: DataFrame, ranges: dict) -> DataFrame:
    """Valores numéricos fora do intervalo válido viram NULL."""
    dtypes = dict(df.dtypes)
    for c, (low, high) in ranges.items():
        if c in dtypes and dtypes[c] in NUMERIC_TYPES:
            df = df.withColumn(c, F.when(F.col(c).between(low, high), F.col(c)))
    return df


# Regras específicas por tabela
RULES = {
    "metacritic_games": {
        "keys":      ["ID"],                       # chave natural -> remove duplicados
        "not_null":  ["ID", "NAME"],               # linhas sem chave/nome são descartadas
        "ranges":    {"METASCORE": (0, 100), "USER_SCORE": (0, 10)},
        "list_cols": ["PUBLISHER"],                # vem como "['Humble Games']"
    },
    "steam_metrics":     {},
    "regional_sales":    {},
    "how_long_to_beat":  {},
    "walkthrough_links": {},
}


def transform_to_silver(df: DataFrame, rules: dict = None):
    """Aplica renome + Data Quality. Devolve (dataframe, relatório de qualidade)."""
    rules = rules or {}
    report = {"linhas_bronze": df.count()}

    # 1) nomes de colunas (de uma vez só, evita conflito de rename em loop)
    new_cols = [apply_name_rules(c) for c in df.columns]
    if len(set(new_cols)) != len(new_cols):
        raise ValueError(f"Renome gerou colunas duplicadas: {new_cols}")
    df = df.toDF(*new_cols)
    df = _safe_drop(df, ["DATETIME_BRONZE", "FILENAME"])      # colunas técnicas do bronze

    # 2) limpeza e tipagem
    df = clean_list_like(df, rules.get("list_cols", []))
    df = clean_strings(df)
    df = auto_cast_types(df)
    df = apply_ranges(df, rules.get("ranges", {}))

    # 3) descarte de linhas inválidas e duplicadas
    not_null = [c for c in rules.get("not_null", []) if c in df.columns]
    if not_null:
        df = df.dropna(subset=not_null)
    report["linhas_sem_chave_removidas"] = report["linhas_bronze"] - df.count()

    keys = [c for c in rules.get("keys", []) if c in df.columns]
    before = df.count()
    df = df.dropDuplicates(keys) if keys else df.dropDuplicates()
    report["duplicadas_removidas"] = before - df.count()
    report["linhas_silver"] = df.count()
    return df, report


# ---------- Escrita ----------
def bronze_to_silver(table: str):
    """Lê bronze.<table>, aplica a transformação e salva silver.<table> (managed table)."""
    src_name = f"{BRONZE_SCHEMA}.{table}"
    df = spark.read.table(f"{CATALOG}.{src_name}")
    df, report = transform_to_silver(df, RULES.get(table, {}))

    # colunas de auditoria
    df = (df
          .withColumn("FILENAME_BRONZE", F.lit(src_name))      # origem rastreável
          .withColumn("FILEDATE_SILVER", F.current_timestamp()))

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")     # permite regravar quando o schema muda (ex.: string -> int)
       .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.{table}"))
    return report

## Executando a transformação para todas as tabelas

O relatório mostra quantas linhas entraram, quantas foram descartadas e quantas chegaram na silver.

In [ ]:
reports = {}
for table in RULES:
    reports[table] = bronze_to_silver(table)
    print(f"{table:20s} {reports[table]}")

## Conferindo os tipos gravados

In [ ]:
for table in RULES:
    print(f"--- silver.{table}")
    spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.{table}").printSchema()

## (SQL) SHOW TABLES IN silver

Verifica os dados gravados no formato Delta Lake tipo MANAGED na camada silver.

In [ ]:
%sql
SHOW TABLES IN silver

## (SQL) DESCRIBE DETAIL nome_tabela;

Detalhes de uma tabela Delta Lake.

In [ ]:
%sql
DESCRIBE DETAIL silver.metacritic_games;

## (SQL) DESCRIBE EXTENDED nome_tabela;

Mostra o tipo da tabela (MANAGED ou EXTERNAL) e o schema.

In [ ]:
%sql
DESCRIBE EXTENDED silver.metacritic_games;